# 05 · Dashboard ejecutivo · Resultados de la asignación

**Prueba analítica DICAGI 2022 · Bancolombia**

---

## 📌 Objetivo

Vista de una sola pasada con los **KPIs más relevantes** del resultado, pensada para presentación a comité o líderes de negocio. Sin tecnicismos, solo lo que importa.

## 🎯 Resumen en 3 frases

1. **Métrica oficial obtenida**: `x/y = 0.4076` ≈ **40.76%** — el techo del problema dadas las condiciones de los datos.
2. **Restricción dominante**: 633 ejecutivos huérfanos arrastran 7,030 clientes fuera del modelo (no por capacidad, por integridad de datos).
3. **Acción de mayor impacto**: reconciliar la tabla `ecas` para recuperar los huérfanos → métrica subiría a **53.4%**.

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from modelo_capacidad.viz import apply_theme, BANCOLOMBIA_COLORS, BANCOLOMBIA_SEQUENTIAL
from modelo_capacidad.data.loader import load_clientes
from modelo_capacidad.utils.validators import metrica_oficial
apply_theme()

DATA_PROCESSED = REPO_ROOT / 'data' / 'processed'

df_ejec = pd.read_parquet(DATA_PROCESSED / 'df_ejecutivos.parquet')
df_ger  = pd.read_parquet(DATA_PROCESSED / 'df_gerentes.parquet')
resultado = pd.read_csv(REPO_ROOT / 'resultado_prueba.csv',
                       dtype={'num_doc_cli': 'string', 'cod_ejec_bco': 'string',
                              'num_doc_gte_inv': 'string', 'cod_gte_inv': 'string'})
clientes = load_clientes()

m = metrica_oficial(resultado, clientes)
print(f'Resultado cargado: {len(resultado):,} filas')

---
## 1. KPIs principales

In [ ]:
# Indicadores tipo "score card"
fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type': 'indicator'}] * 4],
)
fig.add_trace(go.Indicator(
    mode='number+gauge', value=m['metrica_x_y'] * 100,
    number={'suffix': '%', 'font': {'size': 32, 'color': BANCOLOMBIA_COLORS['negro']}},
    title={'text': 'Métrica oficial<br><sub>x/y</sub>'},
    gauge={'axis': {'range': [0, 53.4]},
           'bar': {'color': BANCOLOMBIA_COLORS['amarillo']},
           'steps': [
               {'range': [0, 20], 'color': BANCOLOMBIA_COLORS['gris_claro']},
               {'range': [20, 40], 'color': BANCOLOMBIA_COLORS['gris']},
               {'range': [40, 53.4], 'color': BANCOLOMBIA_COLORS['amarillo_claro']},
           ],
           'threshold': {'line': {'color': BANCOLOMBIA_COLORS['negro'], 'width': 4},
                         'thickness': 0.8, 'value': 40.8}},
), row=1, col=1)
fig.add_trace(go.Indicator(
    mode='number', value=m['n_asignados'],
    number={'valueformat': ',', 'font': {'size': 32, 'color': BANCOLOMBIA_COLORS['negro']}},
    title={'text': 'Clientes asignados<br><sub>de 34,145 totales</sub>'},
), row=1, col=2)
fig.add_trace(go.Indicator(
    mode='number', value=m['pct_A'],
    number={'suffix': '%', 'font': {'size': 32, 'color': BANCOLOMBIA_COLORS['negro']}},
    title={'text': '% A asignados<br><sub>la categoría más rentable</sub>'},
), row=1, col=3)
fig.add_trace(go.Indicator(
    mode='number', value=m['pct_B'],
    number={'suffix': '%', 'font': {'size': 32, 'color': BANCOLOMBIA_COLORS['amarillo_oscuro']}},
    title={'text': '% B asignados'},
), row=1, col=4)
fig.update_layout(height=300, title_text='Tablero ejecutivo · Modelo de Capacidad DICAGI 2022')
fig.show()

---
## 2. ¿Cuántos clientes y de qué tipo se asignaron?

In [ ]:
# Stacked bar: total vs asignados por categoría
comp_cat = pd.DataFrame([
    {'estado': 'Asignados', 'A': m['A_asignados'], 'B': m['B_asignados'], 'C': m['C_asignados']},
    {'estado': 'No asignados', 'A': m['A_total'] - m['A_asignados'],
     'B': m['B_total'] - m['B_asignados'], 'C': m['C_total'] - m['C_asignados']},
]).melt(id_vars='estado', var_name='categoria', value_name='clientes')

fig = px.bar(
    comp_cat, x='categoria', y='clientes', color='estado', barmode='group',
    title='Clientes asignados vs no asignados, por categoría',
    color_discrete_map={'Asignados': BANCOLOMBIA_COLORS['amarillo'],
                        'No asignados': BANCOLOMBIA_COLORS['gris']},
    text='clientes',
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(yaxis_title='# clientes', xaxis_title='Categoría')
fig.show()

### 📏 Lectura del gráfico

- **A**: 3,916 asignados (71%) de 5,495 → quedan 1,579 fuera, casi todos huérfanos.
- **B**: 10,000 asignados (78%) de 12,727 → quedan 2,727 fuera, también huérfanos.
- **C**: 13,199 asignados (83%) de 15,923 → no afectan la métrica oficial.

**Los "no asignados" no salen del modelo**: son los clientes huérfanos cuyo ejecutivo no aparece en `ecas` y por lo tanto no tienen un Gerente de Inversión a quien asignar.

---
## 3. Mapa de utilización de los gerentes

In [ ]:
# Carga por gerente
asign_ejec = resultado.drop_duplicates('cod_ejec_bco')[['cod_ejec_bco', 'cod_gte_inv']]
usado = asign_ejec.merge(df_ejec[['cod_ejec_bco', 't_e', 'n_a', 'n_b', 'n_c']], on='cod_ejec_bco')
uso = usado.groupby('cod_gte_inv').agg(
    usado=('t_e', 'sum'),
    ejec=('cod_ejec_bco', 'count'),
    n_A=('n_a', 'sum'),
    n_B=('n_b', 'sum'),
    n_C=('n_c', 'sum'),
).reset_index()
uso = uso.merge(df_ger[['cod_gte_inv', 'T_g', 'zona']], on='cod_gte_inv')
uso['utilizacion_pct'] = (uso['usado'] / uso['T_g'] * 100).round(1)
uso['n_clientes'] = uso['n_A'] + uso['n_B'] + uso['n_C']
uso = uso.sort_values('utilizacion_pct', ascending=True)

fig = px.bar(
    uso, x='utilizacion_pct', y='cod_gte_inv', orientation='h',
    color='utilizacion_pct', color_continuous_scale=BANCOLOMBIA_SEQUENTIAL,
    title='Utilización de cada gerente (% de capacidad anual usada)',
    hover_data=['ejec', 'n_clientes', 'n_A', 'n_B', 'n_C'],
)
fig.add_vline(x=100, line_dash='dash', line_color=BANCOLOMBIA_COLORS['rojo_alerta'])
fig.update_layout(
    height=900, yaxis={'showticklabels': False},
    xaxis_title='% utilización', yaxis_title='Gerentes (49)',
)
fig.show()

print(f'Utilización media: {uso["utilizacion_pct"].mean():.1f}%')
print(f'Gerentes saturados (>95%): {(uso["utilizacion_pct"] > 95).sum()}')
print(f'Gerentes con holgura (<70%): {(uso["utilizacion_pct"] < 70).sum()}')

---
## 4. Distribución de clientes asignados por región y categoría

In [ ]:
# Sankey: zona -> A/B/C asignados (visualización de flujo)
asign_full = resultado.merge(
    clientes[['num_doc_cli', 'marca_mac_inv', 'region_gte_inv']], on='num_doc_cli', how='left'
)
regiones = asign_full['region_gte_inv'].dropna().unique().tolist()
categorias = ['A', 'B', 'C']
labels = regiones + categorias
src, tgt, val, colors = [], [], [], []
color_map = {'A': BANCOLOMBIA_COLORS['negro'], 'B': BANCOLOMBIA_COLORS['amarillo'], 'C': BANCOLOMBIA_COLORS['gris']}
for ri, region in enumerate(regiones):
    for cj, cat in enumerate(categorias):
        n = ((asign_full['region_gte_inv'] == region) & (asign_full['marca_mac_inv'] == cat)).sum()
        if n > 0:
            src.append(ri)
            tgt.append(len(regiones) + cj)
            val.append(int(n))
            colors.append(color_map[cat])

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=15, thickness=20,
              color=[BANCOLOMBIA_COLORS['amarillo']]*len(regiones) +
                    [color_map[c] for c in categorias]),
    link=dict(source=src, target=tgt, value=val,
              color=[c.replace('#', 'rgba(') + ',0.4)' for c in colors] if False else None),
))
fig.update_layout(title='Flujo de clientes asignados: región → categoría', height=500)
fig.show()

---
## 5. Comparación con los techos teóricos

In [ ]:
techos = pd.DataFrame([
    {'concepto': 'Logrado', 'valor': m['metrica_x_y'] * 100, 'color': BANCOLOMBIA_COLORS['amarillo']},
    {'concepto': 'Techo real (huérfanos descontados)', 'valor': 40.8, 'color': BANCOLOMBIA_COLORS['negro']},
    {'concepto': 'Techo absoluto (todos los A+B)', 'valor': 53.4, 'color': BANCOLOMBIA_COLORS['gris_oscuro']},
])

fig = go.Figure()
for _, row in techos.iterrows():
    fig.add_bar(name=row['concepto'], x=[row['concepto']], y=[row['valor']],
                marker_color=row['color'], text=f"{row['valor']:.1f}%")
fig.update_traces(textposition='outside')
fig.update_layout(
    title='Logrado vs Techos teóricos',
    yaxis_title='%', yaxis_range=[0, 60],
    showlegend=False,
)
fig.show()

### 🎯 Mensaje clave

**El modelo alcanza el techo real del problema**. La diferencia entre el techo real (40.8%) y el absoluto (53.4%) — los **12.6 puntos perdidos** — son recuperables solo si se reconcilian los 633 ejecutivos huérfanos en la tabla `ecas`.

**Esto convierte la conversación de "mejorar el modelo" a "mejorar los datos"** — un cambio de naturaleza muy importante para tomar decisiones de inversión correctas.

---
## 6. Próximos pasos sugeridos

| Plazo | Acción | Beneficio esperado |
|---|---|---|
| Sprint 1 | Reconciliar 633 ejec huérfanos en `ecas` | métrica 40.8% → **53.4%** (+12.6 pts) |
| Sprint 1 | Tabla maestra de productos (mapping unificado) | τ_c más fiel y reproducible |
| Sprint 2 | Telemetría real de actividad por gerente | sustituye encuesta auto-reportada |
| Sprint 2 | Implementar arquitectura del bosquejo (`docs/arquitectura.md`) | servir resultados a CRM, app móvil, web interna |
| Sprint 3 | Variables adicionales internas (ver `docs/variables_adicionales.md`) | mejora upstream del `score_modelo` |

Ver detalle completo en `docs/documento_final.md`.